In [6]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Load user-item interaction matrix
user_item_matrix = pd.read_csv(
    "user_item_matrix.csv",
    index_col=0
)

user_item_matrix.head()


,Biography,Comics,Curtains,Cushions,Dumbbells,Fiction,Foundation,Headphones,Jacket,Jeans,...,Non-fiction,Perfume,Resistance Bands,Shoes,Smartphone,Smartwatch,T-shirt,Treadmill,Wall Art,Yoga Mat
Customer_ID,,,,,,,,,,,,,,,,,,,,,
C1000,1,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
C10000,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
C10001,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
C10002,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
C10003,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
# Transpose to get item-user matrix
item_user_matrix = user_item_matrix.T

# Compute cosine similarity
item_similarity = cosine_similarity(item_user_matrix)

# Convert to DataFrame
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=item_user_matrix.index,
    columns=item_user_matrix.index
)

item_similarity_df.head()

,Biography,Comics,Curtains,Cushions,Dumbbells,Fiction,Foundation,Headphones,Jacket,Jeans,...,Non-fiction,Perfume,Resistance Bands,Shoes,Smartphone,Smartwatch,T-shirt,Treadmill,Wall Art,Yoga Mat
Biography,1.000000,0.000000,0.071281,0.054795,0.066146,0.000000,0.053046,0.066198,0.056835,0.060429,...,0.000000,0.075156,0.079185,0.060371,0.063575,0.072335,0.065476,0.067349,0.069171,0.073496
Comics,0.000000,1.000000,0.061020,0.072969,0.053140,0.000000,0.075061,0.059355,0.075290,0.049975,...,0.000000,0.063100,0.059036,0.079184,0.067389,0.078693,0.071622,0.064010,0.059882,0.072596
Curtains,0.071281,0.061020,1.000000,0.000000,0.045820,0.068260,0.068309,0.071598,0.074424,0.067136,...,0.058351,0.072216,0.063291,0.078234,0.078208,0.076553,0.076528,0.059919,0.000000,0.068267
Cushions,0.054795,0.072969,0.000000,1.000000,0.062278,0.058596,0.066031,0.048265,0.068641,0.081417,...,0.065551,0.073616,0.058544,0.065028,0.068000,0.064831,0.078011,0.087429,0.000000,0.056392
Dumbbells,0.066146,0.053140,0.045820,0.062278,1.000000,0.070253,0.069091,0.064181,0.065654,0.081010,...,0.071085,0.069370,0.000000,0.052028,0.072206,0.065455,0.076411,0.000000,0.076741,0.000000


In [16]:
# Function to generate top-N product recommendations for a given user
# using item-based collaborative filtering.
def recommend_products(user_id, user_item_matrix, similarity_df, top_n=5):

    # Extract the interaction history of the given user
    user_ratings = user_item_matrix.loc[user_id]

    # Identify items the user has already interacted with
    purchased_items = user_ratings[user_ratings > 0].index

    # Compute recommendation scores by averaging similarity scores
    # of items the user has interacted with
    scores = similarity_df[purchased_items].mean(axis=1)

    # Remove already interacted items from recommendations
    scores = scores.drop(purchased_items)

    # Return the top-N highest scoring items
    return scores.sort_values(ascending=False).head(top_n)

# Generate top-5 recommendations for a sample user
recommend_products(
    user_id=user_item_matrix.index[0],
    user_item_matrix=user_item_matrix,
    similarity_df=item_similarity_df
)


,0
Dumbbells,0.073578
Resistance Bands,0.071086
Perfume,0.070530
Lamp,0.070059
Lipstick,0.069468


In [9]:
# Generate top-10 recommendations for the same user to observe
# the effect of changing the recommendation list size
recommend_products(
    user_id=user_item_matrix.index[0],
    user_item_matrix=user_item_matrix,
    similarity_df=item_similarity_df,
    top_n=10
)

,0
Dumbbells,0.073578
Resistance Bands,0.071086
Perfume,0.070530
Lamp,0.070059
Lipstick,0.069468
Curtains,0.069209
Cushions,0.068106
Wall Art,0.067664
Yoga Mat,0.067182
Treadmill,0.067031


In [10]:
#Metric1: Calculate sparsity of the user–item interaction matrix.
sparsity = 1 - (np.count_nonzero(user_item_matrix.values) /
                (user_item_matrix.shape[0] * user_item_matrix.shape[1]))

print("Matrix Sparsity:", round(sparsity, 3))

Matrix Sparsity: 0.917


In [17]:
#Metric3: coverage
# Coverage measures the proportion of unique items
# that appear in the recommendation lists.
all_recommendations = []

for user_id in user_item_matrix.index[:20]:  # test on first 20 users
    recs = recommend_products(
        user_id,
        user_item_matrix,
        item_similarity_df,
        top_n=5
    )
    all_recommendations.extend(recs.index.tolist())

total_items = user_item_matrix.shape[1]
coverage_score = len(set(all_recommendations)) / total_items

print("Coverage:", round(coverage_score, 4))


Coverage: 0.9583


In [18]:
#Metric3: Diversity
# Diversity measures how dissimilar the recommended items are from each other.
def diversity(recommendations, similarity_df):
    sims = similarity_df.loc[recommendations, recommendations]
    upper_triangle = sims.values[np.triu_indices(len(recommendations), k=1)]
    return 1 - upper_triangle.mean()

diversity_scores = []

for user_id in user_item_matrix.index[:20]:
    recs = recommend_products(
        user_id,
        user_item_matrix,
        item_similarity_df,
        top_n=5
    )
    diversity_scores.append(diversity(recs.index.tolist(), item_similarity_df))

print("Average Diversity:", round(np.mean(diversity_scores), 4))


Average Diversity: 0.945
